# Contrastive Predictive Coding (CPC) con xLSTM como modelo autorregresivo

Este notebook corre **el mismo pipeline de CPC** de `01_Pipeline_CPC.ipynb`, pero cambiando el modelo autorregresivo `g_ar` (una GRU en el notebook 01) por una pila **xLSTM**:

- **CPC:** van den Oord, Li & Vinyals (2018), arXiv:1807.03748.
- **Adaptación a sensores corporales:** Haresamudram, Essa & Plötz (2020), arXiv:2012.05333.
- **xLSTM:** Beck, Pöppel, Spanring, Auer, Prudnikova, Kopp, Klambauer, Brandstetter & Hochreiter (2024), *xLSTM: Extended Long Short-Term Memory*, arXiv:2405.04517, https://arxiv.org/html/2405.04517v2

**Qué es igual al notebook 01:** `Encoder`, `Predictor`, `InfoNCE`, `CPCModel`, la carga de datos (resampleo a 50 Hz, ventanas de 200 pasos, z-score por archivo, exclusión de archivos saturados), el split por sujeto (13 y 14 en validación), `BATCH_SIZE`, `N_EPOCHS`, `LR`, `GRAD_CLIP`, `SEED`, las `K` evaluadas, las ventanas etiquetadas y el *linear probe*. Esas celdas son copia de las del notebook 01; ahí están explicadas paso a paso.

**Qué cambia:** la clase `AutoRegressive`, que ahora es una pila xLSTM (sección 3.2), y los nombres de los checkpoints (`cpc_emg_xlstm_K{K}_ep{N}.pt`), para no pisar los de la GRU ni los de la LSTM.

**Requisitos:**

- haber corrido `00_EDA_dataset.ipynb` (genera `_cache/emg_file_stats.csv`);
- ejecutar desde la carpeta `notebooks/`;
- para la comparación final (sección 6), haber corrido las secciones 5–7 de `01_Pipeline_CPC.ipynb`, que generan `_cache/probe_results_gru_lstm.csv`. Si ese archivo no existe, la sección 6 sólo muestra los resultados de xLSTM.

| Sección | Contenido |
| :--- | :--- |
| 1 | ¿Qué es xLSTM? (teoría) |
| 2 | Setup |
| 3 | Pipeline CPC con xLSTM: piezas y verificaciones con datos sintéticos |
| 4 | Entrenamiento sobre sEMG real, con distintas `K` |
| 5 | Evaluación: *accuracy* por horizonte y *linear probe* |
| 6 | Comparación con GRU y LSTM |

# 1. ¿Qué es xLSTM?

xLSTM (Beck, Pöppel, Spanring, Auer, Prudnikova, Kopp, Klambauer, Brandstetter & Hochreiter, 2024; Hochreiter es co-autor de la LSTM original) se pregunta **hasta dónde llega una LSTM si se le aplican las técnicas modernas de los LLMs**. Parte de tres limitaciones de la LSTM:

1. **No puede "revisar" decisiones de almacenamiento.** Si ya guardó algo y luego llega algo más relevante, la compuerta sigmoidal ($i_t, f_t \in (0,1)$) no le deja sobrescribir con fuerza. El paper lo muestra con una tarea de *Nearest Neighbor Search*: recorrer una secuencia buscando el vector más parecido a uno de referencia.
2. **Capacidad de almacenamiento limitada:** toda la información se comprime en una celda **escalar** por unidad (un vector $c^\text{cell}_t \in \mathbb{R}^d$). Por ejemplo, rinde mal al predecir tokens raros.
3. **No es paralelizable:** la dependencia $h_{t-1} \to$ compuertas en $t$ (*memory mixing*) obliga a procesar la secuencia paso a paso, a diferencia de un Transformer.

Para resolverlas propone dos celdas nuevas, **sLSTM** y **mLSTM**, y las empaqueta en **bloques residuales** que se apilan (como las capas de un Transformer).

## 1.1 Idea 1: *exponential gating* con normalizador y estabilizador (sLSTM)

Las compuertas de entrada (y opcionalmente de olvido) pasan de sigmoide a **exponencial**:

$$
\begin{aligned}
\tilde z_t &= W_z x_t + R_z h_{t-1} + b_z, & z_t &= \tanh(\tilde z_t) && \text{entrada a la celda}\\
\tilde i_t &= W_i x_t + R_i h_{t-1} + b_i, & i_t &= \exp(\tilde i_t) && \text{compuerta de entrada exponencial}\\
\tilde f_t &= W_f x_t + R_f h_{t-1} + b_f, & f_t &= \sigma(\tilde f_t)\ \text{ o }\ \exp(\tilde f_t) && \text{compuerta de olvido}\\
\tilde o_t &= W_o x_t + R_o h_{t-1} + b_o, & o_t &= \sigma(\tilde o_t) && \text{compuerta de salida}\\
c_t &= f_t\, c_{t-1} + i_t\, z_t && && \text{celda}\\
n_t &= f_t\, n_{t-1} + i_t && && \text{estado normalizador}\\
h_t &= o_t \odot \tilde h_t, \qquad \tilde h_t = c_t / n_t && && \text{salida normalizada}
\end{aligned}
$$

**Por qué la exponencial:** con $\exp(\tilde i_t)$ una entrada nueva puede pesar *órdenes de magnitud* más que todo lo acumulado y, en la práctica, **sobrescribir** la memoria: esto resuelve la limitación 1. Pero $c_t$ podría crecer sin límite, así que se introduce el **normalizador** $n_t$, que acumula los mismos pesos que $c_t$ pero sin el contenido. Entonces $\tilde h_t = c_t/n_t$ es un **promedio ponderado** de los $z_s$ pasados, con pesos $i_s \prod_{j=s+1}^{t} f_j$, y queda acotado en $[-1, 1]$.

**Estabilizador** $m_t$: $\exp$ de un número grande desborda en `float32` (`exp(89) = inf`). Como $c_t$ y $n_t$ se dividen, se puede restar el mismo valor en el exponente sin cambiar $\tilde h_t$:

$$
m_t = \max\big(\log f_t + m_{t-1},\ \tilde i_t\big), \qquad
i'_t = \exp(\tilde i_t - m_t), \qquad
f'_t = \exp(\log f_t + m_{t-1} - m_t)
$$

y se usan $i'_t, f'_t$ en vez de $i_t, f_t$. El paper demuestra que el resultado es idéntico y que $m_t$ no aporta gradiente. Por eso en el código se hace `.detach()`.

## 1.2 *Memory mixing* y cabezas (sLSTM)

Las matrices recurrentes $R_z, R_i, R_f, R_o$ hacen que $h_{t-1}$ entre en las compuertas del paso $t$: las unidades de memoria **se comunican entre sí** (*memory mixing*). En sLSTM son **bloque-diagonales**: el estado se divide en `num_heads = 4` cabezas y sólo hay mezcla **dentro** de cada cabeza. Esto reduce los parámetros recurrentes de $d^2$ a $d^2/N_h$. Como hay mezcla recurrente, **sLSTM sigue siendo secuencial** (un bucle sobre $t$).

## 1.3 Idea 2: memoria **matricial** (mLSTM)

Para la limitación 2, la celda pasa de un escalar a una **matriz** $C_t \in \mathbb{R}^{d\times d}$ por cabeza, que funciona como una **memoria asociativa clave → valor** (como una capa de atención lineal, pero con compuertas):

$$
\begin{aligned}
q_t &= W_q x_t + b_q, \quad k_t = \tfrac{1}{\sqrt d}\,W_k x_t + b_k, \quad v_t = W_v x_t + b_v && \text{query, key, value}\\
i_t &= \exp(\tilde i_t), \quad f_t = \sigma(\tilde f_t) \text{ o } \exp(\tilde f_t), \quad o_t = \sigma(\tilde o_t) && \text{compuertas (escalares por cabeza)}\\
C_t &= f_t\, C_{t-1} + i_t\, v_t k_t^\top && \text{regla de actualización por covarianza}\\
n_t &= f_t\, n_{t-1} + i_t\, k_t && \text{normalizador (vector)}\\
\tilde h_t &= \frac{C_t\, q_t}{\max\{|n_t^\top q_t|,\, 1\}}, \qquad h_t = o_t \odot \tilde h_t && \text{lectura}
\end{aligned}
$$

- **Escribir:** se suma el producto externo $v_t k_t^\top$ ("guarda el valor $v_t$ bajo la clave $k_t$").
- **Leer:** $C_t q_t = \sum_s w_{ts}\, v_s\,(k_s^\top q_t)$ recupera los valores cuyas claves se parecen a la consulta.
- Con $d = 128$ por cabeza, cada cabeza guarda una matriz de $128\times128$ en vez de un vector de 128: mucha más capacidad.

**Clave:** en mLSTM las compuertas y $q,k,v$ dependen sólo de $x_t$, **no de $h_{t-1}$** (no hay *memory mixing*). Eso permite calcularlo **en paralelo** para toda la secuencia (Apéndice A.3 del paper). Desenrollando la recurrencia:

$$
\tilde h_t = \frac{\sum_{s\le t} D_{ts}\,(q_t^\top k_s)\, v_s}{\max\{|\sum_{s\le t} D_{ts}\, q_t^\top k_s|,\ 1\}}, \qquad
D_{ts} = \exp\Big(\underbrace{\textstyle\sum_{j=s+1}^{t} \log f_j}_{\text{olvido acumulado}} + \tilde i_s\Big)\ \ (s \le t), \quad D_{ts}=0\ \ (s>t)
$$

Es decir, **atención causal** con una matriz de decaimiento $D$ (triangular inferior) en lugar de un softmax. Para $T = 200$ la matriz $T\times T$ es pequeña, así que en entrenamiento usamos esta forma, que es mucho más rápida en MPS que un bucle. En la sección 3.3 se verifica que da **lo mismo** que la forma recurrente.

## 1.4 Bloques xLSTM y arquitectura xLSTM[a:b]

Las celdas se envuelven en **bloques residuales** con *pre-LayerNorm*. Hay dos tipos (Figura 3 y Apéndice, Figs. 10-11):

- **Bloque sLSTM, *post up-projection*** (como un Transformer): LayerNorm → convolución causal (k=4) + Swish para las compuertas i, f → sLSTM de 4 cabezas → GroupNorm por cabeza → residual; luego LayerNorm → **MLP con compuerta** (GeLU, factor 4/3) → residual. Primero resume el pasado en la dimensión original y después proyecta hacia arriba.
- **Bloque mLSTM, *pre up-projection*** (como un *State Space Model* tipo Mamba): LayerNorm → proyección hacia arriba ×2 en dos ramas. En la **rama 1**: convolución causal (k=4) + Swish → $q, k$ (proyecciones bloque-diagonales de bloque 4), mientras que $v$ sale de la rama **sin** convolución → mLSTM → GroupNorm + *skip* aprendible. La **rama 2** es la compuerta de salida externa (Swish). Ambas ramas se multiplican y se proyectan de vuelta a $d$. Proyecta hacia arriba **antes** porque la capacidad de la memoria matricial crece con la dimensión: para $d=512$ con 4 cabezas, cada $C_t$ es de $128\times128$.

La justificación de proyectar a alta dimensión es el **teorema de Cover**: en un espacio de mayor dimensión es más fácil separar linealmente historias distintas. Esto viene muy a cuento en CPC, porque después el `Predictor` usa una transformación **lineal** $W_k c_t$.

Un **xLSTM[a:b]** apila bloques mLSTM y sLSTM en proporción $a:b$; por ejemplo, xLSTM[7:1] tiene 7 mLSTM por cada sLSTM. Aquí usamos **xLSTM[1:1] con 2 bloques** (`("m", "s")`): la misma profundidad que la GRU de 2 capas y un número de parámetros del mismo orden. Una capa lineal inicial lleva $z_t$ (128) a $d_\text{model} = 256$, y una LayerNorm final entrega $c_t$ (256).

## 1.5 Qué esperar en este problema (sin vender humo)

- Los resultados del paper son en **modelado de lenguaje** con secuencias largas y cientos de millones de parámetros. Aquí tenemos ventanas de **T = 200** pasos (4 s), 8 canales y ~1 M de parámetros: no hay garantía de que la ventaja se traslade.
- Lo que **sí** podría ayudar: (a) el *exponential gating* permite "cambiar de opinión" rápido cuando empieza una actividad nueva dentro de la ventana; (b) la memoria matricial y la proyección a alta dimensión pueden dar un $c_t$ más **linealmente separable**, que es justo lo que mide el *linear probe*.
- Riesgos: más **sobreajuste** (pocos sujetos) y más lentitud (el bucle de sLSTM en Python es mucho más lento que la GRU/LSTM fusionadas de PyTorch). Si es demasiado lento, `xlstm_blocks=("m", "m")` (xLSTM[1:0], totalmente paralelo) es la variante que el paper reporta con mejor perplejidad (Tabla 1).

# 2. Setup

Igual que en el notebook 01: detectamos **MPS** (GPU de Apple Silicon) y, si no está disponible, CUDA o CPU.

In [ ]:
import math, time, json
from dataclasses import dataclass, field, asdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)

# 3. Pipeline de CPC con xLSTM

## 3.0 Configuración

`CPCConfig` tiene los **mismos campos y valores** que en el notebook 01, más los parámetros de la pila xLSTM. El tamaño del contexto sigue siendo `c_dim = ar_hidden_size = 256`, así que el `Predictor` no cambia. `ar_num_layers` no se usa aquí: la profundidad la da el número de bloques en `xlstm_blocks`.

In [ ]:
@dataclass
class CPCConfig:
    """
    Hiperparámetros de CPC adaptado a sensores corporales (iguales al notebook 01) + los de la pila xLSTM.

    - n_channels: nº de canales de la señal cruda (depende del dataset)

    Parámetros del g_enc (iguales al notebook 01):
    - enc_channels, enc_kernel_size, enc_dropout

    Parámetros del g_ar (xLSTM, Beck et al., 2024, arXiv:2405.04517):
    - ar_hidden_size: d_model de la pila xLSTM (= dimensión del contexto c_t)
    - ar_dropout: dropout sobre la salida de cada bloque, antes de la suma residual
    - xlstm_blocks: orden de los bloques, "m" = mLSTM, "s" = sLSTM. ("m", "s") es un xLSTM[1:1] de 2 bloques,
                    la misma profundidad que la GRU de 2 capas del notebook 01
    - xlstm_num_heads: nº de cabezas de sLSTM y mLSTM (el paper usa 4)
    - xlstm_conv_kernel: kernel de la convolución causal de cada bloque (el paper usa 4)
    - xlstm_mlstm_proj_factor: up-projection del bloque mLSTM (pre up-projection, el paper usa 2)
    - xlstm_slstm_proj_factor: factor del MLP con compuerta del bloque sLSTM (post up-projection, el paper usa 4/3)

    Predicciones (iguales al notebook 01):
    - n_pred_steps: K, horizontes futuros a predecir
    - n_anchors: instantes de anclaje aleatorios por batch
    """
    # --- entrada ---
    n_channels: int = 6

    # --- g_enc: encoder convolucional ---
    enc_channels: tuple = (32, 64, 128)
    enc_kernel_size: int = 3
    enc_dropout: float = 0.2

    # --- g_ar: pila xLSTM ---
    ar_hidden_size: int = 256
    ar_num_layers: int = 2                  # sin uso en xLSTM; se conserva para que los checkpoints sean comparables
    ar_dropout: float = 0.2
    xlstm_blocks: tuple = ("m", "s")
    xlstm_num_heads: int = 4
    xlstm_conv_kernel: int = 4
    xlstm_mlstm_proj_factor: float = 2.0
    xlstm_slstm_proj_factor: float = 4 / 3

    # --- predicción / pérdida contrastiva ---
    n_pred_steps: int = 12
    n_anchors: int = 4

    @property
    def z_dim(self) -> int:
        """Dimensión del espacio latente z_t: el nº de canales de la última capa del encoder."""
        return self.enc_channels[-1]

    @property
    def c_dim(self) -> int:
        """Dimensión del contexto c_t: el d_model de la pila xLSTM."""
        return self.ar_hidden_size


cfg = CPCConfig(n_channels=6, n_pred_steps=12, n_anchors=4)

# --- Batch SINTÉTICO, sólo para probar las formas de los tensores paso a paso ---
B, T = 32, 200
x_demo = torch.randn(B, cfg.n_channels, T, device=DEVICE)

print("cfg.z_dim =", cfg.z_dim, " cfg.c_dim =", cfg.c_dim)
print("x_demo:", tuple(x_demo.shape), " (B, n_channels, T)")

## 3.1 `g_enc` — el encoder convolucional (igual al notebook 01)

3 bloques `Conv1d(padding='reflect') → ReLU → Dropout` con 32, 64 y 128 canales, sin *stride*: `x (B, C, T) → z (B, 128, T)`.

In [ ]:
class Encoder(nn.Module):
    """
    g_enc: codificador convolucional genérico.

        n_channels: Canales que tiene cada muestra de señal
        x: (B, n_channels, T)  -->  z: (B, z_dim, T)
        k: Tamaño de la ventaba que cada colvolución va a mirar (núm. de pasos hacia el futuro)

    Arquitectura (Haresamudram, Essa & Plötz, 2020, arXiv:2012.05333): una pila
    de bloques  Conv1d -> ReLU -> Dropout, con padding 'reflect' y SIN stride,
    por lo que la longitud temporal T se conserva de principio a fin.

    No depende de ningún dataset: sólo necesita conocer `n_channels` (dado por
    cfg) y recibe T de forma dinámica según el tensor de entrada.
    """

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        k = cfg.enc_kernel_size
        pad = k // 2  # padding "same": conserva la longitud temporal (stride=1) - Rellena los bordes

        in_channels = (cfg.n_channels,) + cfg.enc_channels[:-1]
        blocks = []
        # Esto va a emparejar los bloques de cada capa. 
        # Por ejemplo (6->32), (32->64), (64->128) y comienza con cada una de las capas de la CNN
        # Ésta misma recenta de convolución se va a aplicar más adelante a cada uno de los elementos de Batche y 
        for c_in, c_out in zip(in_channels, cfg.enc_channels):
            blocks.append(nn.Sequential(
                nn.Conv1d(c_in, c_out, kernel_size=k, padding=pad, padding_mode="reflect"),
                nn.ReLU(inplace=True),
                nn.Dropout(cfg.enc_dropout),
            ))
        self.blocks = nn.ModuleList(blocks)

    # Para la generación de los bloques en el proceso de ejecución
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = x
        for block in self.blocks:
            z = block(z)          # (B, C_in, T) -> (B, C_out, T)
        return z                  # (B, z_dim, T)


# --- Demo: verificar formas con el batch sintético x_demo ---
enc = Encoder(cfg).to(DEVICE)
with torch.no_grad():
    z_demo = enc(x_demo)                       # (B, z_dim, T)

n_params = sum(p.numel() for p in enc.parameters())
print(f"Encoder: {n_params/1e3:.1f}k parámetros")
print(f"  entrada  x : {tuple(x_demo.shape)}   (B, n_channels, T)")
print(f"  salida   z : {tuple(z_demo.shape)}   (B, z_dim, T)   <- T se conserva (sin stride)") # Ojo, en el código de Pao sí tiene Stride, por el paper que ella tomó es el de audio

## 3.2 `g_ar` — la pila xLSTM

Ésta es la única pieza que cambia respecto al notebook 01. Se arma en cuatro partes:

1. **Piezas auxiliares:** convolución causal, capa lineal bloque-diagonal y LayerNorm por cabeza.
2. **sLSTM:** la celda y su bloque *post up-projection* (sección 1.1–1.2).
3. **mLSTM:** la celda y su bloque *pre up-projection* (sección 1.3–1.4).
4. **`AutoRegressive`:** la pila de bloques, con la misma interfaz que la GRU: `z (B, L, 128) → c (B, L, 256)`.

**Por qué no usamos el paquete oficial `xlstm`** (github.com/NX-AI/xlstm): sus kernels rápidos están escritos para CUDA/Triton y este notebook corre en MPS (Apple Silicon). Las clases de abajo son una implementación compacta en PyTorch puro, fiel a las ecuaciones y a los bloques del paper (Figuras 10 y 11 del apéndice).

### 3.2.1 Piezas auxiliares

- `CausalConv1d`: convolución *depthwise* con relleno sólo a la izquierda, así $y_t$ depende de $x_{t-3..t}$. Ojito: ésta **no** puede usar el `padding="reflect"` simétrico del `Encoder`, porque miraría el futuro.
- `BlockDiagonalLinear`: una capa lineal "por cabezas", con una matriz pequeña distinta para cada bloque del vector.
- `MultiHeadLayerNorm`: la GroupNorm del paper, que es una LayerNorm independiente para cada cabeza.

In [ ]:
class CausalConv1d(nn.Module):
    """
    Conv1d depthwise (un filtro por canal) CAUSAL: relleno de (kernel - 1) ceros sólo a la izquierda.

        x: (B, L, D)  -->  y: (B, L, D),  con y_t función de x_{t-kernel+1 .. t}
    """

    def __init__(self, dim: int, kernel_size: int):
        super().__init__()
        self.pad = kernel_size - 1
        self.conv = nn.Conv1d(dim, dim, kernel_size, groups=dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.pad(x.transpose(1, 2), (self.pad, 0))   # (B, D, pad + L): ceros sólo en el pasado
        return self.conv(x).transpose(1, 2)            # (B, L, D)


class BlockDiagonalLinear(nn.Module):
    """
    Capa lineal con matriz de pesos bloque-diagonal: el vector de entrada (D) se divide en `n_blocks` trozos y cada
    trozo tiene su propia matriz (D/n_blocks x D/n_blocks). Parámetros: D^2 / n_blocks en vez de D^2.
    """

    def __init__(self, dim: int, n_blocks: int, bias: bool = True):
        super().__init__()
        assert dim % n_blocks == 0, f"dim={dim} no es divisible entre n_blocks={n_blocks}"
        block = dim // n_blocks
        self.n_blocks = n_blocks
        self.weight = nn.Parameter(torch.randn(n_blocks, block, block) / math.sqrt(block))
        self.bias = nn.Parameter(torch.zeros(dim)) if bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unflatten(-1, (self.n_blocks, -1))                       # (..., n_blocks, block)
        y = torch.einsum("...hi,hoi->...ho", x, self.weight).flatten(-2)
        return y if self.bias is None else y + self.bias


class MultiHeadLayerNorm(nn.Module):
    """GroupNorm del paper: LayerNorm independiente sobre cada cabeza, con escala y sesgo aprendibles por dimensión."""

    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unflatten(-1, (self.n_heads, -1))                        # (..., n_heads, head_dim)
        x = F.layer_norm(x, x.shape[-1:]).flatten(-2)
        return x * self.weight + self.bias

### 3.2.2 sLSTM: celda y bloque (*post up-projection*)

`sLSTMCell.forward` implementa directamente las ecuaciones de la sección 1.1 con el estabilizador. Para que el bucle sobre $t$ sea lo más barato posible, todo lo que **no** depende de $h_{t-1}$ (las proyecciones de entrada $W x_t$) se calcula antes del bucle, para toda la secuencia a la vez. Dentro del bucle sólo queda el *memory mixing* $R\,h_{t-1}$, que es bloque-diagonal por cabeza.

Inicialización (como en el código oficial): el sesgo de la compuerta de olvido va de 3 a 6 a lo largo de las unidades, así que $\sigma(3..6) \approx 0.95..0.998$ y la memoria **arranca reteniendo** información.

Estado inicial: $c_0 = n_0 = h_0 = 0$ y $m_0 = -\infty$. En $t=1$ queda $m_1 = \tilde i_1$, $i'_1 = 1$ y $n_1 = 1$. A partir de ahí, por inducción, $n_t \ge 1$ siempre, así que la división $c_t/n_t$ nunca es $0/0$.

In [ ]:
def bias_linspace_(bias: torch.Tensor, start: float, end: float):
    """Inicializa un sesgo con valores equiespaciados en [start, end] (sesgo de la compuerta de olvido)."""
    with torch.no_grad():
        bias.copy_(torch.linspace(start, end, bias.numel()))


class sLSTMCell(nn.Module):
    """
    Celda sLSTM con exponential gating, normalizador n_t, estabilizador m_t y memory mixing por cabezas
    (Beck et al., 2024, Sección 2.2). Recurrente: hace un bucle explícito sobre el tiempo.

        x_conv, x: (B, L, D)  -->  h: (B, L, D)
        (x_conv alimenta las compuertas i, f; x alimenta z, o, como en la Figura 10 del paper)
    """

    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads, self.head_dim = n_heads, dim // n_heads
        # W: entrada -> pre-activaciones de z, i, f, o (bloque-diagonal por cabeza, con sesgo)
        self.W = nn.ModuleDict({g: BlockDiagonalLinear(dim, n_heads) for g in "zifo"})
        # R: h_{t-1} -> pre-activaciones (memory mixing), una matriz (head_dim x head_dim) por compuerta y cabeza
        self.R = nn.Parameter(torch.randn(4, n_heads, self.head_dim, self.head_dim) / math.sqrt(self.head_dim))
        bias_linspace_(self.W["f"].bias, 3.0, 6.0)

    def forward(self, x_conv: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        B, L, _ = x.shape
        H, hd = self.n_heads, self.head_dim
        # Parte de las compuertas que NO depende de h_{t-1}: se calcula para todo t de una vez -> (B, L, 4, H, hd)
        wx = torch.stack([self.W["z"](x), self.W["i"](x_conv), self.W["f"](x_conv), self.W["o"](x)], dim=2)
        wx = wx.unflatten(-1, (H, hd))

        h = x.new_zeros(B, H, hd)
        c = x.new_zeros(B, H, hd)
        n = x.new_zeros(B, H, hd)
        m = x.new_full((B, H, hd), float("-inf"))
        outs = []
        for t in range(L):
            rh = torch.einsum("bhi,ghoi->bgho", h, self.R)            # memory mixing R h_{t-1}: (B, 4, H, hd)
            z_pre, i_pre, f_pre, o_pre = (wx[:, t] + rh).unbind(dim=1)

            log_f = F.logsigmoid(f_pre)                                 # f_t = sigmoid(f~) -> log f_t
            m_new = torch.maximum(log_f + m, i_pre).detach()            # estabilizador (sin gradiente)
            i = torch.exp(i_pre - m_new)                                # i'_t
            f = torch.exp(log_f + m - m_new)                            # f'_t

            c = f * c + i * torch.tanh(z_pre)                           # celda
            n = f * n + i                                               # normalizador
            h = torch.sigmoid(o_pre) * (c / n)                          # salida normalizada
            m = m_new
            outs.append(h)
        return torch.stack(outs, dim=1).flatten(-2)                     # (B, L, D)


class sLSTMBlock(nn.Module):
    """
    Bloque residual sLSTM con post up-projection (Figura 10 del paper):

        x -> LN -> [conv causal + Swish para i, f] -> sLSTM (4 cabezas) -> GroupNorm -> (+x)
          -> LN -> MLP con compuerta (GeLU, factor 4/3) -> (+)
    """

    def __init__(self, dim: int, cfg: CPCConfig):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.conv = CausalConv1d(dim, cfg.xlstm_conv_kernel)
        self.cell = sLSTMCell(dim, cfg.xlstm_num_heads)
        self.cell_norm = MultiHeadLayerNorm(dim, cfg.xlstm_num_heads)

        hidden = round(cfg.xlstm_slstm_proj_factor * dim)
        self.ffn_norm = nn.LayerNorm(dim)
        self.ffn_up = nn.Linear(dim, 2 * hidden)         # dos ramas: valor y compuerta
        self.ffn_down = nn.Linear(hidden, dim)
        self.dropout = nn.Dropout(cfg.ar_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.norm(x)
        y_conv = F.silu(self.conv(y))
        x = x + self.dropout(self.cell_norm(self.cell(y_conv, y)))

        value, gate = self.ffn_up(self.ffn_norm(x)).chunk(2, dim=-1)
        return x + self.dropout(self.ffn_down(F.gelu(gate) * value))

### 3.2.3 mLSTM: celda y bloque (*pre up-projection*)

`mLSTMCell` tiene **dos** implementaciones de las mismas ecuaciones (sección 1.3):

- `forward`: la forma **paralela**, con la matriz de decaimiento $D$ y una "atención" causal. Es la que se usa para entrenar.
- `forward_recurrent`: la forma **recurrente** con la memoria $C_t$ explícita, paso a paso. Es más lenta y sólo se usa para comprobar, en 3.3, que ambas dan el mismo resultado.

En la forma paralela el estabilizador es $m_t = \max_{s\le t} \log D_{ts}$ (el máximo de cada fila), y la cota inferior del denominador pasa de $1$ a $e^{-m_t}$, porque numerador y denominador quedaron multiplicados por $e^{-m_t}$.

Las compuertas $i, f$ son **escalares por cabeza** y se calculan a partir de $[q, k, v]$. Inicialización como en el código oficial: pesos en cero, sesgo de $f$ de 3 a 6 y sesgo de $i \sim \mathcal N(0, 0.1)$.

In [ ]:
class mLSTMCell(nn.Module):
    """
    Celda mLSTM con memoria matricial C_t (head_dim x head_dim por cabeza) y regla de covarianza
    (Beck et al., 2024, Sección 2.3 y Apéndice A.3). Sin memory mixing -> paralelizable.

        q, k, v: (B, L, D)  -->  h~: (B, L, D)   (la compuerta de salida o_t la aplica el bloque)
    """

    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads, self.head_dim = n_heads, dim // n_heads
        self.igate = nn.Linear(3 * dim, n_heads)     # i~_t, f~_t: un escalar por cabeza
        self.fgate = nn.Linear(3 * dim, n_heads)
        nn.init.zeros_(self.igate.weight)
        nn.init.normal_(self.igate.bias, mean=0.0, std=0.1)
        nn.init.zeros_(self.fgate.weight)
        bias_linspace_(self.fgate.bias, 3.0, 6.0)

    def _heads(self, q, k, v):
        """(B, L, D) -> (B, H, L, hd) para q, k, v, y pre-activaciones de compuertas (B, H, L)."""
        qkv = torch.cat([q, k, v], dim=-1)
        i_pre = self.igate(qkv).transpose(1, 2)
        log_f = F.logsigmoid(self.fgate(qkv)).transpose(1, 2)            # f_t = sigmoid(f~) -> log f_t
        split = lambda x: x.unflatten(-1, (self.n_heads, self.head_dim)).transpose(1, 2)
        return split(q), split(k), split(v), i_pre, log_f

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        """Forma PARALELA (usada para entrenar)."""
        q, k, v, i_pre, log_f = self._heads(q, k, v)
        L = q.shape[2]

        # log D_ts = sum_{j=s+1..t} log f_j + i~_s  = F_t - F_s + i~_s,  con F_t = sum_{j<=t} log f_j
        F_cum = torch.cumsum(log_f, dim=-1)                                # (B, H, L)
        log_D = F_cum[..., :, None] - F_cum[..., None, :] + i_pre[..., None, :]
        causal = torch.ones(L, L, dtype=torch.bool, device=q.device).tril()
        log_D = log_D.masked_fill(~causal, float("-inf"))                  # D_ts = 0 si s > t (el futuro no cuenta)

        m = log_D.amax(dim=-1, keepdim=True).detach()                      # estabilizador m_t: (B, H, L, 1)
        D = torch.exp(log_D - m)

        S = (q @ k.transpose(-1, -2)) / math.sqrt(self.head_dim) * D       # (B, H, L, L): D_ts * q_t . k_s / sqrt(d)
        denom = torch.maximum(S.sum(dim=-1, keepdim=True).abs(), torch.exp(-m))   # max(|n_t^T q_t|, 1), estabilizado
        h = (S / denom) @ v                                                # (B, H, L, hd)
        return h.transpose(1, 2).flatten(-2)                               # (B, L, D)

    def forward_recurrent(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        """Forma RECURRENTE, con la memoria matricial C_t explícita (sólo para verificar la forma paralela)."""
        q, k, v, i_pre, log_f = self._heads(q, k, v)
        B, H, L, hd = q.shape
        k = k / math.sqrt(hd)

        C = q.new_zeros(B, H, hd, hd)                                      # memoria matricial
        n = q.new_zeros(B, H, hd)                                          # normalizador
        m = q.new_full((B, H), float("-inf"))                              # estabilizador
        outs = []
        for t in range(L):
            m_new = torch.maximum(log_f[..., t] + m, i_pre[..., t])
            i = torch.exp(i_pre[..., t] - m_new)[..., None]                # (B, H, 1)
            f = torch.exp(log_f[..., t] + m - m_new)[..., None]
            q_t, k_t, v_t = q[:, :, t], k[:, :, t], v[:, :, t]

            C = f[..., None] * C + i[..., None] * (v_t[..., :, None] * k_t[..., None, :])   # C_t = f C + i v k^T
            n = f * n + i * k_t                                                             # n_t = f n + i k
            num = (C @ q_t[..., None]).squeeze(-1)                                          # C_t q_t
            den = torch.maximum((n * q_t).sum(-1, keepdim=True).abs(), torch.exp(-m_new)[..., None])
            outs.append(num / den)
            m = m_new
        return torch.stack(outs, dim=2).transpose(1, 2).flatten(-2)       # (B, L, D)


class mLSTMBlock(nn.Module):
    """
    Bloque residual mLSTM con pre up-projection (Figura 11 del paper):

        x -> LN -> up-proj (x2) -> rama mLSTM: conv causal + Swish -> q, k  (v sin conv) -> mLSTM -> GroupNorm
                                                  + skip aprendible
                                -> rama compuerta: Swish
           -> (rama mLSTM * compuerta) -> down-proj -> (+x)
    """

    def __init__(self, dim: int, cfg: CPCConfig):
        super().__init__()
        inner = int(cfg.xlstm_mlstm_proj_factor * dim)
        n_qkv_blocks = inner // 4                      # proyecciones bloque-diagonales con bloques de tamaño 4
        self.norm = nn.LayerNorm(dim)
        self.up = nn.Linear(dim, 2 * inner)
        self.conv = CausalConv1d(inner, cfg.xlstm_conv_kernel)
        self.q = BlockDiagonalLinear(inner, n_qkv_blocks)
        self.k = BlockDiagonalLinear(inner, n_qkv_blocks)
        self.v = BlockDiagonalLinear(inner, n_qkv_blocks)
        self.cell = mLSTMCell(inner, cfg.xlstm_num_heads)
        self.cell_norm = MultiHeadLayerNorm(inner, cfg.xlstm_num_heads)
        self.skip = nn.Parameter(torch.ones(inner))   # skip aprendible
        self.down = nn.Linear(inner, dim)
        self.dropout = nn.Dropout(cfg.ar_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_m, x_gate = self.up(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv(x_m))
        h = self.cell(self.q(x_conv), self.k(x_conv), self.v(x_m))
        h = self.cell_norm(h) + self.skip * x_conv
        h = h * F.silu(x_gate)                         # compuerta de salida externa o_t
        return x + self.dropout(self.down(h))

### 3.2.4 `AutoRegressive`: la pila de bloques como `g_ar`

$z_t$ (128) → `Linear` → $d_\text{model}=256$ → bloques en el orden de `cfg.xlstm_blocks` → `LayerNorm` → $c_t$ (256).

La clase se llama `AutoRegressive`, igual que la GRU del notebook 01, para que `CPCModel` (sección 3.5) se pueda copiar sin cambiar ni una línea.

In [ ]:
class AutoRegressive(nn.Module):
    """
    g_ar con una pila xLSTM (Beck et al., 2024). Misma interfaz que la GRU del notebook 01:

        z: (B, L, z_dim)  -->  c: (B, L, c_dim),   c_dim = cfg.ar_hidden_size

    Causal: todas sus piezas lo son (conv con relleno sólo a la izquierda, sLSTM recurrente, mLSTM con máscara
    triangular, y LayerNorm/Linear que actúan sobre cada instante por separado). Se verifica en la sección 3.3.
    """

    BLOCKS = {"m": mLSTMBlock, "s": sLSTMBlock}

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        d = cfg.ar_hidden_size
        self.proj_in = nn.Linear(cfg.z_dim, d)
        self.blocks = nn.ModuleList([self.BLOCKS[b](d, cfg) for b in cfg.xlstm_blocks])
        self.norm_out = nn.LayerNorm(d)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.proj_in(z)
        for block in self.blocks:
            x = block(x)
        return self.norm_out(x)


# --- Demo: verificar formas encadenando con el Encoder ---
ar = AutoRegressive(cfg).to(DEVICE)
with torch.no_grad():
    c_demo = ar(z_demo.transpose(1, 2))        # (B, z_dim, T) -> (B, T, z_dim) -> (B, T, c_dim)

print(f"AutoRegressive (xLSTM {cfg.xlstm_blocks}): {sum(p.numel() for p in ar.parameters())/1e3:.0f}k parámetros")
print(f"  entrada  z : {tuple(z_demo.transpose(1, 2).shape)}   (B, L, z_dim)")
print(f"  salida   c : {tuple(c_demo.shape)}   (B, L, c_dim)")

## 3.3 Verificaciones de `g_ar` antes de entrenar

1. **Causalidad:** se perturba $z$ **sólo** a partir del instante $t_0 = 100$. Si $c_{<100}$ cambia aunque sea un poco, el modelo está mirando el futuro y el InfoNCE sería trampa. La diferencia en $c_{<100}$ debe ser 0, y en $c_{\ge100}$ claramente mayor que 0.
2. **mLSTM paralelo = mLSTM recurrente:** la forma con la matriz $D$ debe dar el mismo $\tilde h$ que la recurrencia con la memoria $C_t$ explícita. Se compara en `float64`, donde deben coincidir hasta ~1e-10. En `float32` las dos formas suman en distinto orden y difieren por redondeo (~1e-4 relativo), lo cual es normal.

In [ ]:
torch.manual_seed(SEED)
z_check = torch.randn(4, 200, cfg.z_dim, device=DEVICE)
z_pert = z_check.clone()
z_pert[:, 100:] += torch.randn_like(z_pert[:, 100:])

with torch.no_grad():
    ar_check = AutoRegressive(cfg).to(DEVICE).eval()
    diff = (ar_check(z_check) - ar_check(z_pert)).abs()
print(f"Causalidad: máx |Δc| antes de t0 = {diff[:, :100].max().item():.2e}   |   desde t0 = {diff[:, 100:].max().item():.2e}")
assert diff[:, :100].max().item() < 1e-5, "c_t depende del futuro"

# En float64 y CPU (MPS no soporta float64): así la comparación no se contamina con redondeo de float32
with torch.no_grad():
    cell = mLSTMCell(dim=64, n_heads=4).double()
    nn.init.normal_(cell.igate.weight, std=0.5)       # compuertas no triviales para que la prueba sea exigente
    nn.init.normal_(cell.fgate.weight, std=0.5)
    q_c, k_c, v_c = torch.randn(3, 2, 50, 64, dtype=torch.float64).unbind(0)
    gap = (cell(q_c, k_c, v_c) - cell.forward_recurrent(q_c, k_c, v_c)).abs().max().item()
print(f"mLSTM (float64): máx |paralelo - recurrente| = {gap:.2e}")
assert gap < 1e-8

## 3.4 Predicciones log-bilineales `W_k` e `InfoNCE` (iguales al notebook 01)

- `Predictor`: una transformación lineal sin sesgo $W_k$ (`c_dim → z_dim`) por cada horizonte $k = 1..K$.
- `InfoNCE`: para cada $k$, matriz de scores $S_{ij} = (W_k c_t^{(i)}) \cdot z_{t+k}^{(j)}$ y `cross_entropy(S, arange(B))`, con negativos del minibatch (Haresamudram et al., 2020).

In [ ]:
class Predictor(nn.Module):
    """
    Cabezas de predicción log-bilineal: una transformación lineal W_k por cada
    horizonte futuro k = 1..K, sin sesgo.

        pred_k(c_t) = W_k @ c_t     (aproxima z_{t+k} en el espacio latente)

    El score de CPC (Ec. 3) es f_k(x_{t+k}, c_t) = exp(z_{t+k}^T W_k c_t): esta
    clase sólo calcula W_k c_t; el producto punto contra los candidatos z_{t+k}
    y el softmax los hace la clase InfoNCE.

    Genérica respecto al dataset: sólo depende de c_dim, z_dim y K (cfg).
    """

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        self.n_pred_steps = cfg.n_pred_steps # Definición de los K pasos al futuro

        # Creación de una W_k diferente para cada paso k en el futuro
        self.heads = nn.ModuleList([
            nn.Linear(cfg.c_dim, cfg.z_dim, bias=False) # Matriz de forma (z_dim, c_dim). Pasa de c_t -> vector del mismo tamaño que z_t
            for _ in range(cfg.n_pred_steps) 
        ])

    def forward(self, c_t: torch.Tensor) -> torch.Tensor:
        """
        c_t: (B, c_dim) — contexto en UN instante de anclaje fijo
        -> (K, B, z_dim): una predicción W_k c_t por cada horizonte k
        """
        return torch.stack([head(c_t) for head in self.heads], dim=0) # Esto hace c_t @ head.weight.T


# --- Demo: predicciones a partir del contexto en un instante fijo ---
pred = Predictor(cfg).to(DEVICE)
with torch.no_grad():
    preds_demo = pred(c_demo[:, 20, :])
print("predicciones W_k c_t :", tuple(preds_demo.shape), " (K, B, z_dim)")

In [ ]:
class InfoNCE(nn.Module):
    """
    Pérdida contrastiva de CPC (Ec. 4), con negativos de minibatch, para todos
    los horizontes k = 1..K de una sola vez.

    No depende de ningún dataset: opera únicamente sobre los tensores que
    producen Encoder/AutoRegressive/Predictor. La estrategia de negativos es
    la del paper (Haresamudram et al., 2020): para cada horizonte k, el
    positivo de la secuencia i es z_{t+k}^(i) y los negativos son los
    z_{t+k}^(j) (j != i) de las OTRAS secuencias del mismo batch, en el mismo
    instante t+k.
    """

    def __init__(self, n_pred_steps: int):
        super().__init__()
        self.n_pred_steps = n_pred_steps # K, pasos en el futuro que debo de dar

    def forward(self, preds: torch.Tensor, z: torch.Tensor, t: int):
        """
        preds : (K, B, z_dim)  salidas de Predictor evaluadas en c_t
        z     : (B, L, z_dim)  secuencia completa de latentes (para leer z_{t+k})
        t     : instante de anclaje usado para calcular `preds`

        Devuelve (loss_mean, loss_per_k, acc_per_k); estos dos últimos son
        tensores (K,), uno por horizonte. `loss_per_k` SÍ conserva el grafo de
        autograd (necesario para que CPCModel.info_nce pueda acumularla sobre
        varios anclajes y aun así hacer backward()); `acc_per_k` va detached
        porque el argmax que la calcula no es diferenciable de todos modos.
        """
        K, B, _ = preds.shape
        device = preds.device
        target = torch.arange(B, device=device)     # la clase correcta de la fila i es i

        loss_per_k = torch.zeros(K, device=device) # Inicializamos el tensor de pérdidas
        acc_per_k = torch.zeros(K, device=device) # Inicializamos el tensor de accuracy

        for k in range(1, K + 1):
            z_future = z[:, t + k, :]                # (B, z_dim): candidatos z_{t+k}
            logits = preds[k - 1] @ z_future.t()     # (B, B): S_ij = pred_i . z_{t+k}^(j)
            loss_per_k[k - 1] = F.cross_entropy(logits, target)
            acc_per_k[k - 1] = (logits.argmax(dim=1) == target).float().mean()

        return loss_per_k.mean(), loss_per_k, acc_per_k.detach()


# --- Demo: pérdida en un anclaje fijo (con pesos aleatorios debe rondar log(B) y la accuracy, 1/B) ---
t_anchor = 20
info_nce_loss = InfoNCE(n_pred_steps=cfg.n_pred_steps).to(DEVICE)
with torch.no_grad():
    _, loss_per_k, acc_per_k = info_nce_loss(pred(c_demo[:, t_anchor, :]), z_demo.transpose(1, 2), t_anchor)
print(f"loss media = {loss_per_k.mean().item():.4f}   (log B = {math.log(B):.4f})")
print("acc_per_k  :", acc_per_k.cpu().numpy().round(3))

## 3.5 `CPCModel` — composición de las 4 piezas (igual al notebook 01)

`CPCModel` construye `self.ar = AutoRegressive(cfg)`, que en este notebook es la pila xLSTM. Por eso la clase se copia sin cambios.

In [ ]:
class CPCModel(nn.Module):
    """
    Modelo CPC completo: Encoder (g_enc) + AutoRegressive (g_ar) + Predictor
    (W_k) + InfoNCE, siguiendo la adaptación a sensores corporales de
    Haresamudram, Essa & Plötz (2020, arXiv:2012.05333).

    Al estar compuesto enteramente por clases genéricas, este modelo no
    conoce nada sobre el dataset de origen: basta con ajustar `cfg.n_channels`
    para entrenarlo sobre EMG, IMU, audio o cualquier otra señal 1D
    multicanal, pasándole tensores (B, n_channels, T).
    """

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder = Encoder(cfg)
        self.ar = AutoRegressive(cfg)
        self.predictor = Predictor(cfg)
        self.info_nce_loss = InfoNCE(cfg.n_pred_steps)

    def encode(self, x: torch.Tensor):
        """x: (B, n_channels, T) -> z: (B, T, z_dim), c: (B, T, c_dim)"""
        z = self.encoder(x).transpose(1, 2)   # (B, z_dim, T) -> (B, T, z_dim)
        c = self.ar(z)                        # (B, T, c_dim)
        return z, c

    def info_nce(self, x: torch.Tensor, anchors=None):
        """
        Codifica x y calcula la pérdida InfoNCE promediada sobre uno o varios
        instantes de anclaje aleatorios (cfg.n_anchors si `anchors` es None).
        """
        cfg = self.cfg
        z, c = self.encode(x)
        B, L, _ = z.shape
        K = cfg.n_pred_steps

        lo, hi = max(1, L // 8), L - K         # deja sitio para poder leer z_{t+K}
        if hi <= lo:
            raise ValueError(f"Ventana T={L} demasiado corta para K={K} pasos de predicción")
        if anchors is None:
            anchors = torch.randint(lo, hi, size=(cfg.n_anchors,)).tolist()

        losses, accs = [], []
        for t in anchors:
            preds = self.predictor(c[:, t, :])                    # (K, B, z_dim)
            _, loss_per_k, acc_per_k = self.info_nce_loss(preds, z, t)
            losses.append(loss_per_k)
            accs.append(acc_per_k)

        loss_per_k = torch.stack(losses).mean(dim=0)   # promedio sobre anclajes, por k
        acc_per_k = torch.stack(accs).mean(dim=0)
        return loss_per_k.mean(), loss_per_k, acc_per_k


# --- Demo: correr el pipeline completo sobre el batch sintético x_demo ---
model = CPCModel(cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
loss, loss_per_k, acc_per_k = model.info_nce(x_demo)

print(f"CPCModel: {n_params/1e6:.2f} M parámetros")
print(f"  entrada x                : {tuple(x_demo.shape)}   (B, n_channels, T)")
print(f"  InfoNCE loss (promedio)  : {loss.item():.4f}")
print(f"  loss por horizonte k     : {loss_per_k.detach().cpu().numpy().round(3)}")
print(f"  accuracy por horizonte k : {acc_per_k.detach().cpu().numpy().round(3)}")

# 4. Entrenamiento sobre el dataset real (sEMG del HAR dataset)

Mismo preprocesamiento que la sección 4 del notebook 01, cuyas decisiones están justificadas ahí y en `00_EDA_dataset.ipynb`:

- **Señal:** `Norm_RMS_1..8`.
- **Exclusiones:** sólo trials (sin MVC) y sin los archivos con canales saturados.
- **Ventanas:** resampleo a `TARGET_FS = 50` Hz, z-score por canal dentro de cada archivo y ventanas de 200 pasos (4 s) con 50% de solape.
- **Validación:** los sujetos 13 y 14 se reservan completos.

In [ ]:
import re
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# --- Rutas (mismo layout que en 00_EDA_dataset.ipynb) ---
DATA_ROOT = Path.cwd().resolve().parent / "datos_unzipped"
EMG_DIR = DATA_ROOT / "Data"
CACHE_DIR = Path.cwd() / "_cache"
assert EMG_DIR.exists(), f"No encontrado: {EMG_DIR} (ejecuta este notebook desde notebooks/)"

# --- Catálogo de archivos sEMG (mismo patrón de nombre que en el EDA) ---
EMG_RE = re.compile(
    r"Subj_(?P<subject>\d+)_(?:(?P<mvc>MVC)_)?(?P<task>Isokin|MMH_Bim|MMH_One|All)_(?P<side>[LR])"
    r"(?:_(?P<load_kg>\d+)kg_(?P<condition>St|\d+bpm|Trial_\d))?_EMG_data\.csv"
)


def parse_emg_files(paths):
    rows = []
    for p in sorted(paths):
        m = EMG_RE.fullmatch(p.name)
        if m is None:
            raise ValueError(f"Nombre de archivo inesperado: {p.name}")
        d = m.groupdict()
        rows.append(dict(
            file=p.name, path=p, subject=int(d["subject"]), task=d["task"],
            is_mvc=d.get("mvc") is not None,
        ))
    return pd.DataFrame(rows)


emg = parse_emg_files(EMG_DIR.glob("Subj_*/*.csv"))
trials = emg[~emg.is_mvc].copy()   # sólo trials: las grabaciones MVC no son secuencias de actividad

# --- Excluir los archivos con canales saturados (sección 6 del EDA): std > 50 mV en algún canal ---
STD_LIMIT_MV = 50
STATS_CSV = CACHE_DIR / "emg_file_stats.csv"
assert STATS_CSV.exists(), (
    f"No encontrado: {STATS_CSV}. Ejecuta primero 00_EDA_dataset.ipynb (sección 6) "
    "para generar el caché de estadísticas de señal."
)
file_stats = pd.read_csv(STATS_CSV)
std_cols = [f"raw_std_{c}" for c in range(1, 9)]
file_stats["std_max_mV"] = file_stats[std_cols].max(axis=1) * 1e3
BAD_EMG_FILES = set(file_stats.loc[file_stats.std_max_mV > STD_LIMIT_MV, "file"])

trials = trials[~trials.file.isin(BAD_EMG_FILES)].reset_index(drop=True)
print(f"{len(trials)} trials sEMG utilizables ({len(BAD_EMG_FILES)} excluidos por canales saturados)")
print("Sujetos:", sorted(trials.subject.unique()))

## 4.1 `EMGWindowDataset` — resampleo, ventaneo y split por sujeto (igual al notebook 01)

In [ ]:
TARGET_FS = 50          # Hz: frecuencia común tras resampleo (Norm_RMS ya es una envolvente suave de 250 ms)
WINDOW_SECONDS = 4.0
WINDOW_LEN = int(TARGET_FS * WINDOW_SECONDS)   # 200 pasos, igual T que x_demo en la sección 3
HOP_LEN = WINDOW_LEN // 2                      # 50% de solape entre ventanas consecutivas

NORM_RMS = [f"Norm_RMS_{i}" for i in range(1, 9)]
VAL_SUBJECTS = {13, 14}   # sujetos completos reservados para validación (nunca mezclados con train)


def load_resampled(path, target_fs=TARGET_FS, return_grid=False):
    """
    Lee Norm_RMS_1..8 de un trial y los interpola a una rejilla uniforme de `target_fs` Hz.
    Con return_grid=True devuelve también `t` (tiempo de cada fila del CSV) y `t_grid`, necesarios
    para llevar las etiquetas de `Labels/` (índices de fila) a la rejilla remuestreada (sección 5).
    """
    d = pd.read_csv(path, usecols=["Time"] + NORM_RMS)
    t = (d["Time"].to_numpy() - d["Time"].iloc[0]) / 1000.0   # segundos desde el inicio del trial
    t_grid = np.arange(0, t[-1], 1.0 / target_fs)
    sig = np.stack([np.interp(t_grid, t, d[c].to_numpy()) for c in NORM_RMS], axis=0).astype(np.float32)  # (8, T)
    return (sig, t, t_grid) if return_grid else sig


def zscore_channels(sig):
    """z-score por canal, con media/std calculadas dentro del propio archivo."""
    mu, sd = sig.mean(axis=1, keepdims=True), sig.std(axis=1, keepdims=True)
    return (sig - mu) / np.clip(sd, 1e-6, None)


class EMGWindowDataset(Dataset):
    """
    Ventanas (n_channels=8, WINDOW_LEN) recortadas de las grabaciones sEMG reales, ya resampleadas
    a TARGET_FS y normalizadas por canal. No usa `Labels/`: CPC es self-supervised.
    """

    def __init__(self, catalog, window_len=WINDOW_LEN, hop_len=HOP_LEN, verbose=True):
        self.signals = []     # una señal (8, T_i) por archivo válido
        self.index = []       # (file_idx, start) de cada ventana

        for i, row in enumerate(catalog.itertuples()):
            sig = zscore_channels(load_resampled(row.path))      # (8, T_i)
            if sig.shape[1] < window_len:
                continue                                          # trial demasiado corto, se descarta
            self.signals.append(torch.from_numpy(sig))
            for start in range(0, sig.shape[1] - window_len + 1, hop_len):
                self.index.append((len(self.signals) - 1, start))
            if verbose and (i + 1) % 100 == 0:
                print(f"  leídos {i + 1}/{len(catalog)} archivos...")

        self.window_len = window_len

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        file_idx, start = self.index[idx]
        sig = self.signals[file_idx]
        return sig[:, start:start + self.window_len]   # (n_channels, window_len)


t0 = time.time()
train_catalog = trials[~trials.subject.isin(VAL_SUBJECTS)]
val_catalog = trials[trials.subject.isin(VAL_SUBJECTS)]

print(f"Train: {len(train_catalog)} trials (sujetos {sorted(train_catalog.subject.unique())})")
train_ds = EMGWindowDataset(train_catalog)
print(f"Val:   {len(val_catalog)} trials (sujetos {sorted(val_catalog.subject.unique())})")
val_ds = EMGWindowDataset(val_catalog, verbose=False)

print(f"\n{len(train_ds)} ventanas de train, {len(val_ds)} de val  ({time.time() - t0:.0f} s)")
print("una ventana:", tuple(train_ds[0].shape), " (n_channels, WINDOW_LEN)")

## 4.2 `DataLoader`s, parámetros y tiempo por paso

Mismo `BATCH_SIZE = 64` que en el notebook 01. La celda también mide cuánto tarda un paso de entrenamiento (forward + backward) con la pila xLSTM, para estimar la duración de la sección 4.4.

Como referencia, en un Mac con MPS se midió:

| `g_ar` | Parámetros de `g_ar` | s / paso |
| :--- | :--- | :--- |
| GRU (notebook 01) | ~691 k | ~0.16 |
| xLSTM `("m", "s")` | ~850 k | ~0.70 |
| xLSTM `("m", "m")` | — | ~0.33 |

Casi todo el tiempo extra es el bucle paso a paso del bloque sLSTM: 200 pasos con varias operaciones pequeñas cada uno, y en MPS cada operación tiene un costo fijo de lanzamiento.

In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
if len(val_loader) == 0:
    print(f"[aviso] val_loader quedó vacío ({len(val_ds)} ventanas < BATCH_SIZE={BATCH_SIZE}); baja BATCH_SIZE.")


def time_train_step(model, n_reps=3):
    """Segundos por paso de entrenamiento (forward + backward) con un batch sintético de tamaño real."""
    x = torch.randn(BATCH_SIZE, 8, WINDOW_LEN, device=DEVICE)
    model.train()
    for rep in range(n_reps + 1):                     # la primera repetición es de calentamiento
        if rep == 1:
            if DEVICE.type == "mps": torch.mps.synchronize()
            t0 = time.time()
        model.info_nce(x)[0].backward()
        model.zero_grad()
    if DEVICE.type == "mps": torch.mps.synchronize()
    return (time.time() - t0) / n_reps


torch.manual_seed(SEED)
model_tmp = CPCModel(CPCConfig(n_channels=8, n_pred_steps=12)).to(DEVICE)
sec_per_step = time_train_step(model_tmp)
print(f"CPCModel (xLSTM, K=12): {sum(p.numel() for p in model_tmp.parameters())/1e6:.2f} M parámetros "
      f"(g_ar: {sum(p.numel() for p in model_tmp.ar.parameters())/1e3:.0f}k)")
print(f"train_loader: {len(train_loader)} batches de {BATCH_SIZE}  |  val_loader: {len(val_loader)} batches")
print(f"{sec_per_step:.2f} s/paso  ->  ~{sec_per_step * len(train_loader) / 60:.1f} min por época de train "
      f"(~{sec_per_step * len(train_loader) * 25 / 3600:.1f} h para 25 épocas, por cada K)")
del model_tmp

## 4.3 Entrenamiento (self-supervised) y checkpoints

`run_epoch` y `train_cpc` son los del notebook 01 (sección 4.3). Igual que allá, `train_cpc` siempre crea un modelo nuevo con semilla fija, recorta la norma del gradiente y se detiene si la pérdida deja de ser finita.

Los checkpoints se llaman `cpc_emg_xlstm_K{K}_ep{N_EPOCHS}.pt`, para no pisar los de la GRU (`cpc_emg_K…`) ni los de la LSTM (`cpc_emg_lstm_K…`).

In [ ]:
N_EPOCHS = 25
LR = 2e-4
GRAD_CLIP = 5.0   # norma máxima del gradiente; evita que un batch raro haga diverger la corrida


def run_epoch(model, optimizer, loader, train: bool):
    model.train(mode=train)
    tot_loss, tot_acc, n_batches = 0.0, 0.0, 0
    for x in loader:
        x = x.to(DEVICE)
        with torch.set_grad_enabled(train):
            loss, loss_per_k, acc_per_k = model.info_nce(x)
            loss_value = loss.item()
            if not math.isfinite(loss_value):
                raise RuntimeError(f"Pérdida no finita ({loss_value}) en el batch {n_batches}: el entrenamiento divergió")
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
        tot_loss += loss_value
        tot_acc += acc_per_k.mean().item()
        n_batches += 1
    return tot_loss / n_batches, tot_acc / n_batches


def train_cpc(cfg: CPCConfig, n_epochs=N_EPOCHS, lr=LR):
    """Entrena un CPCModel NUEVO (pesos aleatorios con semilla fija) y devuelve (model, history)."""
    torch.manual_seed(SEED)
    model = CPCModel(cfg).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    t0 = time.time()
    for epoch in range(1, n_epochs + 1):
        train_loss, train_acc = run_epoch(model, optimizer, train_loader, train=True)
        val_loss, val_acc = run_epoch(model, optimizer, val_loader, train=False)
        history["train_loss"].append(train_loss); history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
        print(f"[K={cfg.n_pred_steps}] epoch {epoch:>2}/{n_epochs}  "
              f"train loss {train_loss:.4f} acc {train_acc:.3f}  |  "
              f"val loss {val_loss:.4f} acc {val_acc:.3f}  "
              f"({time.time() - t0:.0f} s transcurridos)")
    return model, history

In [ ]:
CKPT_DIR = Path.cwd() / "_checkpoints"
CKPT_DIR.mkdir(exist_ok=True)


def ckpt_path(cfg: CPCConfig, n_epochs=N_EPOCHS) -> Path:
    return CKPT_DIR / f"cpc_emg_xlstm_K{cfg.n_pred_steps}_ep{n_epochs}.pt"


def save_cpc(model, cfg: CPCConfig, history, n_epochs=N_EPOCHS) -> Path:
    path = ckpt_path(cfg, n_epochs)
    torch.save({
        "model_state_dict": model.state_dict(),
        "cfg": asdict(cfg),              # dict simple: se puede cargar con torch.load(weights_only=True)
        "history": history,
        "target_fs": TARGET_FS,
        "window_len": WINDOW_LEN,
    }, path)
    return path


def load_cpc(path: Path):
    ckpt = torch.load(path, map_location=DEVICE)
    cfg = CPCConfig(**ckpt["cfg"])
    model = CPCModel(cfg).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    return model, cfg, ckpt["history"]

## 4.4 Un modelo xLSTM por `K`

Mismas `K_VALUES` que la sección 5.1 del notebook 01. Para cada `K` se busca su checkpoint; si no existe, se entrena y se guarda.

**Ojo con el tiempo:** con `("m", "s")` son ~1.5 h por `K` en MPS (se midieron ~3.4 min por época de train, más la validación), unas **10 h para las 7 `K`**. Cada `K` se guarda al terminar, así que se puede interrumpir y al volver a correr la celda sólo se entrenan las que falten. Dos formas de acelerar:

- empezar con menos `K`, por ejemplo `K_VALUES = [12, 20]`, que son las mejores de la GRU;
- usar `xlstm_blocks=("m", "m")` en `CPCConfig` (xLSTM[1:0], ~2× más rápido). En ese caso conviene cambiar también el nombre del checkpoint en `ckpt_path`, para no mezclar variantes.

In [ ]:
K_VALUES = [4, 6, 8, 12, 16, 20, 24]   # mismas K que el notebook 01

cpc_models = {}      # K -> CPCModel entrenado
cpc_histories = {}   # K -> history de su entrenamiento
for K in K_VALUES:
    cfg_k = CPCConfig(n_channels=8, n_pred_steps=K, n_anchors=4)
    path = ckpt_path(cfg_k)
    if path.exists():
        model, _, hist = load_cpc(path)
        print(f"K={K:>2}: cargado {path.name}")
    else:
        print(f"K={K:>2}: no existe {path.name}, entrenando {N_EPOCHS} épocas...")
        model, hist = train_cpc(cfg_k)
        save_cpc(model, cfg_k, hist)
    cpc_models[K] = model.eval()
    cpc_histories[K] = hist

print()
for K, hist in cpc_histories.items():
    print(f"K={K:>2}  última época: train acc {hist['train_acc'][-1]:.3f} | val acc {hist['val_acc'][-1]:.3f}"
          f"  (promedio sobre k=1..{K})")

## 4.5 Curvas de entrenamiento por `K`

Mismo formato que la sección 4.4 del notebook 01. Se guardan en `_figures/CPC_DS1_xLSTM_Perdida_acc_k_{K}.png`.

In [ ]:
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})

FIG_DIR = Path.cwd() / "_figures"
FIG_DIR.mkdir(exist_ok=True)

for K, hist in cpc_histories.items():
    epochs = range(1, len(hist["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    fig.suptitle(f"Entrenamiento CPC (xLSTM) — K = {K}", fontweight="bold")

    axes[0].plot(epochs, hist["train_loss"], color="#4c72b0", marker="o", ms=4, label="train")
    axes[0].plot(epochs, hist["val_loss"], color="#dd8452", marker="o", ms=4, label="val")
    axes[0].set(xlabel="epoch", ylabel="InfoNCE loss", title="Pérdida (menor es mejor)")
    axes[0].legend()

    axes[1].plot(epochs, hist["train_acc"], color="#4c72b0", marker="o", ms=4, label="train")
    axes[1].plot(epochs, hist["val_acc"], color="#dd8452", marker="o", ms=4, label="val")
    axes[1].axhline(1 / BATCH_SIZE, color="#9e9e9e", ls="--", lw=1, label=f"azar (1/{BATCH_SIZE})")
    axes[1].set(xlabel="epoch", ylabel="accuracy (promedio sobre k)", title="Recuperar el positivo entre negativos")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(FIG_DIR / f"CPC_DS1_xLSTM_Perdida_acc_k_{K}.png", dpi=120)
    plt.show()

# 5. Evaluación downstream

Mismo protocolo que la sección 5 del notebook 01:

- **5.1** — *accuracy* de InfoNCE **por horizonte `k`**;
- **5.2** — ventanas etiquetadas con las 7 clases de `Labels/`;
- **5.3** — *linear probe*: `Encoder` + xLSTM congelados, se extrae $c_t$ del último instante y se entrena una regresión logística.

Las líneas base son la **misma arquitectura xLSTM sin preentrenar** y las **features crudas**.

## 5.1 *Accuracy* de InfoNCE por horizonte `k`

In [ ]:
@torch.no_grad()
def acc_per_horizon(model, loader):
    """Accuracy de InfoNCE en `loader` para cada horizonte k = 1..K (anclajes aleatorios con semilla fija)."""
    model.eval()
    torch.manual_seed(SEED)
    accs = [model.info_nce(x.to(DEVICE))[2].cpu() for x in loader]
    return torch.stack(accs).mean(dim=0).numpy()


acc_k = {K: acc_per_horizon(m, val_loader) for K, m in cpc_models.items()}

fig, ax = plt.subplots(figsize=(7, 3.6))
for K, acc in acc_k.items():
    ax.plot(np.arange(1, K + 1), acc, marker="o", ms=4, label=f"K={K}")
ax.axhline(1 / BATCH_SIZE, color="#9e9e9e", ls="--", lw=1, label=f"azar (1/{BATCH_SIZE})")
ax.set(xlabel="horizonte k (pasos de 1/TARGET_FS s)", ylabel="accuracy (val)",
       title="InfoNCE en validación, por horizonte", ylim=(0, 1))
ax.legend()
fig.tight_layout()

pd.DataFrame({f"K={K}": pd.Series(acc, index=np.arange(1, K + 1)) for K, acc in acc_k.items()}).T.round(3)

## 5.2 Ventanas etiquetadas (igual al notebook 01)

In [ ]:
LABEL_DIR = DATA_ROOT / "Labels"
LABEL_ORDER = ["N", "LF", "K", "PT", "LT", "W", "PF"]   # mismas 7 clases y orden que en el EDA
EVAL_HOP = 25   # pasos entre ventanas evaluadas (0.5 s a 50 Hz)


def label_file(row):
    return LABEL_DIR / row.path.parent.name / row.file.replace("_EMG_data.csv", "_EMG_labels.csv")


def labels_are_consistent(lab):
    """Misma regla del EDA: ningún segmento vacío/invertido ni solapado con el siguiente."""
    gap_to_next = lab.Start_Frame.shift(-1) - lab.End_Frame
    return not ((lab.End_Frame < lab.Start_Frame).any() or (gap_to_next < 0).any())


labeled_trials = []   # (subject, señal z-score (8, T) como tensor, clase por paso (T,) con -1 = sin etiqueta)
bad_label_files = []
t0 = time.time()
for row in trials.itertuples():
    lab = pd.read_csv(label_file(row))
    if not labels_are_consistent(lab):
        bad_label_files.append(row.file)
        continue
    sig, t, t_grid = load_resampled(row.path, return_grid=True)
    if sig.shape[1] < WINDOW_LEN:
        continue
    y_rows = np.full(len(t), -1, dtype=np.int64)       # clase de cada fila del CSV original
    for s0, s1, name in lab[["Start_Frame", "End_Frame", "Label"]].itertuples(index=False):
        y_rows[s0:s1 + 1] = LABEL_ORDER.index(name)
    y = y_rows[np.clip(np.searchsorted(t, t_grid), 0, len(t) - 1)]   # clase de cada paso de la rejilla
    labeled_trials.append((row.subject, torch.from_numpy(zscore_channels(sig)), y))

print(f"{len(labeled_trials)} trials etiquetados ({time.time() - t0:.0f} s); "
      f"excluidos por etiquetas inconsistentes: {bad_label_files}")


def trial_windows(sig, y):
    """Ventanas (n, 8, WINDOW_LEN) que terminan cada EVAL_HOP pasos y su clase en el último instante."""
    windows = sig.unfold(1, WINDOW_LEN, EVAL_HOP).permute(1, 0, 2)   # (n_win, 8, WINDOW_LEN)
    ends = WINDOW_LEN - 1 + EVAL_HOP * np.arange(windows.shape[0])
    keep = y[ends] >= 0
    return windows[torch.from_numpy(keep)], y[ends][keep]


# Clase y sujeto de cada ventana: no dependen del modelo, se calculan una sola vez
Y, G = [], []
for subject, sig, y in labeled_trials:
    _, yy = trial_windows(sig, y)
    Y.append(yy)
    G.append(np.full(len(yy), subject))
Y, G = np.concatenate(Y), np.concatenate(G)
is_test = np.isin(G, list(VAL_SUBJECTS))

print(f"{len(Y)} ventanas etiquetadas: {(~is_test).sum()} train (sujetos 1-12), {is_test.sum()} test {sorted(VAL_SUBJECTS)}")
pd.DataFrame({"train": pd.Series(Y[~is_test]).value_counts(), "test": pd.Series(Y[is_test]).value_counts()}
             ).rename(index=dict(enumerate(LABEL_ORDER))).reindex(LABEL_ORDER).fillna(0).astype(int).T

## 5.3 *Linear probe* con el xLSTM congelado vs. líneas base

Se agrega un detalle respecto al notebook 01: la línea base aleatoria es un **xLSTM aleatorio**. Una pila xLSTM aleatoria ya proyecta a alta dimensión, así que podría ser una línea base más fuerte que una GRU aleatoria. Lo que importa es **cuánto mejora el preentrenamiento sobre su propia arquitectura**.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


@torch.no_grad()
def window_features(feature_fn, batch_size=512):
    """Aplica feature_fn a todas las ventanas etiquetadas, en el mismo orden que Y y G."""
    feats = []
    for _, sig, y in labeled_trials:
        windows, _ = trial_windows(sig, y)
        for i in range(0, len(windows), batch_size):
            feats.append(feature_fn(windows[i:i + batch_size]).cpu())
    return torch.cat(feats).numpy()


def cpc_context(model):
    """c_t en el último instante de la ventana, con el modelo congelado."""
    model.eval()
    return lambda x: model.encode(x.to(DEVICE))[1][:, -1, :]


def raw_stats(x):
    """Media y std de cada canal en el último segundo de la ventana."""
    last = x[..., -TARGET_FS:]
    return torch.cat([last.mean(dim=-1), last.std(dim=-1)], dim=1)


def linear_probe(F):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    clf.fit(F[~is_test], Y[~is_test])
    pred = clf.predict(F[is_test])
    per_class = f1_score(Y[is_test], pred, labels=range(len(LABEL_ORDER)), average=None, zero_division=0)
    return {"F1 macro": f1_score(Y[is_test], pred, average="macro"),
            "balanced acc": balanced_accuracy_score(Y[is_test], pred),
            **{f"F1 {name}": f for name, f in zip(LABEL_ORDER, per_class)}}


torch.manual_seed(SEED)
representations = {f"CPC xLSTM K={K}": cpc_context(m) for K, m in cpc_models.items()}
representations["xLSTM aleatorio"] = cpc_context(CPCModel(CPCConfig(n_channels=8, n_pred_steps=K_VALUES[0])).to(DEVICE))
representations["Features crudas"] = raw_stats

probe_results = {}
for name, feature_fn in representations.items():
    t0 = time.time()
    probe_results[name] = linear_probe(window_features(feature_fn))
    print(f"{name:<18} F1 macro {probe_results[name]['F1 macro']:.3f}   ({time.time() - t0:.0f} s)")

probe_df = pd.DataFrame(probe_results).T
probe_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
colors = ["#4c72b0"] * len(cpc_models) + ["#9e9e9e", "#c7c7c7"]
bars = ax.bar(probe_df.index, probe_df["F1 macro"], color=colors)
ax.bar_label(bars, fmt="%.3f", fontsize=9)
ax.set(ylabel="F1 macro (sujetos de test)", title="Linear probe sobre la representación congelada (xLSTM)",
       ylim=(0, 1))
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()

# 6. Comparación con GRU y LSTM

Los resultados de xLSTM se guardan en `_cache/probe_results_xlstm.csv`, con el mismo formato que `_cache/probe_results_gru_lstm.csv` (sección 7.5 del notebook 01). Si ese archivo existe, se cargan los resultados de GRU y LSTM y se comparan las tres arquitecturas. Todas usan los mismos datos, `K`, épocas, semilla y protocolo de evaluación: **la única diferencia es `g_ar`**.

| Columna | Qué es |
| :--- | :--- |
| `val acc InfoNCE` | *accuracy* de la tarea auxiliar en validación (última época); sólo comparable **a la misma `K`** |
| `brecha train-val` | *accuracy* de train − val en InfoNCE; si es grande, hay sobreajuste |
| `F1 macro` | *linear probe* sobre los sujetos de test: **criterio principal** |
| `ganancia vs aleatorio` | F1 macro − F1 de la misma arquitectura sin preentrenar |

In [ ]:
def resumen_probe(g_ar, histories, probe_df, prefix, random_name):
    """Una fila por K (CPC preentrenado) + una fila para la versión aleatoria de la misma arquitectura."""
    f1_random = probe_df.loc[random_name, "F1 macro"]
    rows = []
    for K, hist in histories.items():
        rows.append({"g_ar": g_ar, "representación": "CPC", "K": K,
                     "train acc InfoNCE": hist["train_acc"][-1], "val acc InfoNCE": hist["val_acc"][-1],
                     "brecha train-val": hist["train_acc"][-1] - hist["val_acc"][-1],
                     **probe_df.loc[f"{prefix} K={K}"].to_dict()})
    rows.append({"g_ar": g_ar, "representación": "aleatorio", "K": np.nan, **probe_df.loc[random_name].to_dict()})
    df = pd.DataFrame(rows)
    df["ganancia vs aleatorio"] = df["F1 macro"] - f1_random
    return df


resultados_xlstm = resumen_probe("xLSTM", cpc_histories, probe_df, "CPC xLSTM", "xLSTM aleatorio")
resultados_xlstm.to_csv(CACHE_DIR / "probe_results_xlstm.csv", index=False)
print("Guardado:", CACHE_DIR / "probe_results_xlstm.csv")

GRU_LSTM_CSV = CACHE_DIR / "probe_results_gru_lstm.csv"
if GRU_LSTM_CSV.exists():
    comparacion = pd.concat([pd.read_csv(GRU_LSTM_CSV), resultados_xlstm], ignore_index=True)
else:
    print(f"[aviso] No existe {GRU_LSTM_CSV.name}: corre las secciones 5-7 de 01_Pipeline_CPC.ipynb para comparar.")
    comparacion = pd.concat([resultados_xlstm, pd.DataFrame([{
        "g_ar": "—", "representación": "features crudas", "K": np.nan, **probe_df.loc["Features crudas"].to_dict()}])],
        ignore_index=True)

cols = ["g_ar", "representación", "K", "val acc InfoNCE", "brecha train-val", "F1 macro", "balanced acc",
        "ganancia vs aleatorio"]
comparacion[cols].round(3)

## 6.1 F1 macro y *accuracy* de InfoNCE en función de `K`

- **Izquierda:** F1 macro del *linear probe*. Las líneas punteadas son la versión aleatoria de cada arquitectura y la línea gris discontinua, las features crudas.
- **Derecha:** *accuracy* de InfoNCE en validación (última época).

In [ ]:
ARCH_COLORS = {"GRU": "#4c72b0", "LSTM": "#55a868", "xLSTM": "#c44e52"}
cpc_rows = comparacion[comparacion["representación"] == "CPC"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for g_ar, d in cpc_rows.groupby("g_ar"):
    color = ARCH_COLORS.get(g_ar)
    axes[0].plot(d.K, d["F1 macro"], color=color, marker="o", label=f"CPC {g_ar}")
    axes[1].plot(d.K, d["val acc InfoNCE"], color=color, marker="o", label=f"CPC {g_ar}")
    is_random = (comparacion["g_ar"] == g_ar) & (comparacion["representación"] == "aleatorio")
    if is_random.any():
        axes[0].axhline(comparacion.loc[is_random, "F1 macro"].item(), color=color, ls=":", lw=1.2,
                        label=f"{g_ar} aleatorio")
axes[0].axhline(probe_df.loc["Features crudas", "F1 macro"], color="#9e9e9e", ls="--", lw=1, label="features crudas")

axes[0].set(xlabel="K (horizontes de predicción en el preentrenamiento)", ylabel="F1 macro (sujetos de test)",
            title="Linear probe: utilidad de c_t para reconocer actividades")
axes[1].set(xlabel="K", ylabel="accuracy InfoNCE (val, última época)", title="Tarea auxiliar contrastiva", ylim=(0, 1))
for ax in axes:
    ax.set_xticks(K_VALUES)
    ax.legend(fontsize=8)
fig.tight_layout()

f1_por_k = cpc_rows.pivot(index="K", columns="g_ar", values="F1 macro").rename(index=int)
for other in ["GRU", "LSTM"]:
    if other in f1_por_k:
        f1_por_k[f"xLSTM − {other}"] = f1_por_k["xLSTM"] - f1_por_k[other]
f1_por_k.round(3)

## 6.2 F1 por clase con la mejor `K` de cada arquitectura

In [ ]:
best = cpc_rows.loc[cpc_rows.groupby("g_ar")["F1 macro"].idxmax()]
class_cols = [f"F1 {name}" for name in LABEL_ORDER]

fig, ax = plt.subplots(figsize=(11, 3.8))
width = 0.8 / len(best)
x = np.arange(len(LABEL_ORDER))
for j, (_, row) in enumerate(best.iterrows()):
    offset = (j - (len(best) - 1) / 2) * width
    bars = ax.bar(x + offset, row[class_cols].astype(float), width, color=ARCH_COLORS.get(row["g_ar"]),
                  label=f"{row['g_ar']} (K={int(row['K'])}, F1 macro {row['F1 macro']:.3f})")
    ax.bar_label(bars, fmt="%.2f", fontsize=7)
ax.set_xticks(x, LABEL_ORDER)
ax.set(ylabel="F1 (sujetos de test)", title="F1 por clase — mejor K de cada g_ar", ylim=(0, 1))
ax.legend(fontsize=8)
fig.tight_layout()

best.set_index("g_ar")[["K", "F1 macro"] + class_cols].round(3)

## 6.3 Cómo leer los resultados

- **Criterio principal:** el F1 macro del *linear probe*, no la *accuracy* de InfoNCE.
- **xLSTM ≈ GRU/LSTM** (diferencias ≤ 0.02–0.03, que con 2 sujetos de test pueden ser ruido): con ventanas de 4 s y ~1 M de parámetros, la memoria matricial y el *exponential gating* no aportan. La GRU es más barata y suficiente.
- **xLSTM gana en InfoNCE pero no en el *probe*:** la memoria extra se usa para resolver mejor la tarea contrastiva (posiblemente detalles de corto plazo), no para separar actividades. Una salida es subir `K`.
- **xLSTM gana en el *probe* y sobre su propia versión aleatoria:** la proyección a alta dimensión y la memoria matricial dan un $c_t$ más linealmente separable. Los siguientes pasos serían probar `("m", "m")` o `("m", "m", "m", "s")` (xLSTM[3:1]) y ventanas más largas.
- **Brecha train–val más grande que la de la GRU:** sobreajuste. Prueba subir `ar_dropout` o usar menos bloques.
- **xLSTM aleatorio ya alto:** parte de la ventaja viene de la arquitectura, no del preentrenamiento. Mira la columna `ganancia vs aleatorio`.
- Para una conclusión firme se necesitaría validación cruzada *leave-one-subject-out*, como en el notebook 01.